In [29]:
# !pip install tensorly
# !pip install tensorly-torch

In [30]:
import torch
import torch.nn as nn
class reshape(nn.Module):
    '''
    reshapes the 3-order tensor into 6-order tensor

    ----------
    split : list
        split indices to be applied to each mode of the 3-order tensor

    map_type : int
        based on attached - 1 or compressed - 2 splitting method

    device : str
        operation device, default value is cpu


    inputs a 3-order torch.tensor

    returns a 6-order torch.tensor
    '''
    def __init__(self, split, map_type=1, device='cpu'):
        super(reshape, self).__init__()

        self.split = split
        self.map_type = map_type
        self.device = device

    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []

        if self.map_type == 1:
            # Approach 1 : Attached
            C_indices, H_indices, W_indices = [
                [sum(dim // self.split[i] for _ in range(j)) for j in range(self.split[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]

            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1],
                                           H_indices[j]:H_indices[j+1],
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            # Approach 2 : Compressed
            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            C_stride_indices = torch.arange(i, C, self.split[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.split[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.split[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)

        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.split[0] * self.split[1] * self.split[2])
        result = torch.cat(chunks).view(
            batch_size, self.split[0], self.split[1], self.split[2],
            *chunks[0].shape[1:])
        return result

    def forward(self, x):
        chunks = self.split_into_chunks(x)
        output = self.stack_chunks_to_form_tensor(chunks)
        return output

In [31]:
import torch
import torch.nn as nn
import tensorly as tl
from tltorch import TRL, TCL
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from einops import rearrange

In [32]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [33]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 8

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [34]:
def topk_accuracy(outputs, targets, topk=(1,)):
    '''
    calculates top-k accuracy

    ----------
    outputs : torch.tensor

    targets : torch.tensor

    topk  : tuple
        calculates top k accuracy given outpurs and targets


    reutrns a python dictionary of top i <= k accuracies

    '''
    maxk = max(topk)
    _, topk_indices = torch.topk(input=outputs, k=maxk, dim=1, largest=True, sorted=True)
    correct = topk_indices.eq(targets.view(-1, 1).expand_as(topk_indices))
    accuracies = {}
    for k in topk:
        correct_k = correct[:,:k].float().sum()
        accuracies[k] = {'correct':correct_k, 'accuracy': (correct_k / outputs.shape[0]) * 100.0}
    return accuracies

In [35]:
def cp(module):
  return sum(p.numel() for p in module.parameters())

In [36]:
def print_gpu_memory_usage(stage):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    string = f'{stage} - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB'
    print(string)
    return string


In [37]:
def append_to_file(file_name, text):
    with open(file_name, 'a') as file:
        file.write(text + '\n')

# FC layers

In [38]:
class CNN1(nn.Module):
    def __init__(self):
        super(CNN1, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8, 256, bias = False)
        self.fc2 = nn.Linear(256,10, bias = False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model1 = CNN1().to(device)


In [39]:
classifier1 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256, bias = False),
    nn.Linear(256, 10, bias = False)
)

print(cp(classifier1))
append_to_file(file_name='TRL_report.txt', text=f'FC classifier # parameters {cp(classifier1)}')

1051136


In [40]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters())

In [41]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model1.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model1(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s
          
        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model1.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [42]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TRL_report.txt', text=f'FC took {end_time - start_time} time')
append_to_file(file_name='TRL_report.txt', text=f'FC had {string}')
append_to_file(file_name='TRL_report.txt', text=f'FC last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

total forward time : 1.0833728313446045, total backward time : 2.1294705867767334
Train epoch 1: top1=0.5599200129508972%, top2=0.753279983997345%, top3=0.8492599725723267%, top4=0.9071599841117859%, top5=0.9435799717903137%, loss=0.15388202280029656, time=7.366296768188477s
Test epoch 1: top1=0.6345999836921692%, top2=0.8208000063896179%, top3=0.9003999829292297%, top4=0.9408999681472778%, top5=0.967799961566925%, loss=0.1282458631783724, time=1.0299952030181885s
Memory Usage  - Allocated: 48.91 MB, Reserved: 70.00 MB
total forward time : 1.068629503250122, total backward time : 2.0630505084991455
Train epoch 2: top1=0.6969999670982361%, top2=0.8559799790382385%, top3=0.9220799803733826%, top4=0.9581199884414673%, top5=0.9768999814987183%, loss=0.10752107179522515, time=7.104956865310669s
Test epoch 2: top1=0.6879000067710876%, top2=0.8495000004768372%, top3=0.9177999496459961%, top4=0.9517999887466431%, top5=0.9736999869346619%, loss=0.11228602046221495, time=

In [43]:
append_to_file(file_name='TRL_report.txt', text=f'########################################')

# TRL from Tensorly

In [44]:
class CNN2(nn.Module):
    def __init__(self):
        super(CNN2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.trl = TRL(input_shape = (64,2,2), output_shape = (10), factorization='tucker', rank=(10,1,1,10))
        # self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = x.reshape(x.size(0), 64,2,2)
        x = self.trl(x)
        return x


model2 = CNN2().to(device)


In [45]:
classifier2 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256),
    TRL(input_shape = (64,2,2), output_shape = (10), factorization='tucker', rank=(10,1,1,10))
)

print(cp(classifier2))
append_to_file(file_name='TRL_report.txt', text=f'TRL tensorly classifier # parameters {cp(classifier2)}')

1049676


In [46]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters())

In [47]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model2.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model2(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s
        
        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model2.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [48]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TRL_report.txt', text=f'TRL tensorly took {end_time - start_time} time')
append_to_file(file_name='TRL_report.txt', text=f'TRL tensorly had {string}')
append_to_file(file_name='TRL_report.txt', text=f'TRL tensorly last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

total forward time : 1.9206652641296387, total backward time : 2.933920383453369
Train epoch 1: top1=0.5189799666404724%, top2=0.7174800038337708%, top3=0.8178199529647827%, top4=0.8824399709701538%, top5=0.9264599680900574%, loss=0.16724742201060055, time=8.881155252456665s
Test epoch 1: top1=0.6100000143051147%, top2=0.7949000000953674%, top3=0.8798999786376953%, top4=0.9309999942779541%, top5=0.9572999477386475%, loss=0.13909459157139062, time=1.1324777603149414s
Memory Usage  - Allocated: 48.90 MB, Reserved: 70.00 MB
total forward time : 1.9343526363372803, total backward time : 2.9919474124908447
Train epoch 2: top1=0.6549400091171265%, top2=0.826479971408844%, top3=0.9016199707984924%, top4=0.9454799890518188%, top5=0.9691799879074097%, loss=0.12221514031171798, time=9.022599935531616s
Test epoch 2: top1=0.6699999570846558%, top2=0.8349999785423279%, top3=0.9080999493598938%, top4=0.9465999603271484%, top5=0.9693999886512756%, loss=0.11907824716567993, tim

In [49]:
append_to_file(file_name='TRL_report.txt', text=f'########################################')

# TRL Method 1 just 3D tensors

In [50]:
import math
class TRL2(nn.Module):
    def __init__(self, input_shape, output_shape, rank, bias = False): # last rank for the output shape
          super(TRL2, self).__init__()
          self.G = nn.Parameter(torch.empty(rank), requires_grad=True)
          nn.init.kaiming_uniform_(self.G, a=math.sqrt(5))
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)
          self.fc4 = nn.Linear(rank[3], output_shape[0], bias = bias)

    def forward(self, x):
          G = self.fc4(self.G)
          x = self.fc3(x)
          x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = rearrange(x, 'b w h c -> b c h w')
          x = torch.einsum('b x y z , x y z d -> b d', x, G)

          return x

In [51]:
class CNN3(nn.Module):
    def __init__(self):
        super(CNN3, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.trl = TRL2(input_shape = (64,2,2), output_shape = (10,), rank=(10,1,1,10))
        # self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = x.reshape(x.size(0), 64,2,2)
        x = self.trl(x)
        return x


model3 = CNN3().to(device)


In [52]:
classifier3 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256),
    TRL2(input_shape = (64,4,4), output_shape = (10, ), rank=(10,1,1,10))
)

print(cp(classifier3))
append_to_file(file_name='TRL_report.txt', text=f'TRL method 1 classifier # parameters {cp(classifier3)}')

1049680


In [53]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model3.parameters())

In [54]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model3.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model3(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s
        

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model3.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [55]:
n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TRL_report.txt', text=f'TRL method 1 took {end_time - start_time} time')
append_to_file(file_name='TRL_report.txt', text=f'TRL method 1 had {string}')
append_to_file(file_name='TRL_report.txt', text=f'TRL method 1 last epoch result:\n{report_train}\n{report_test}')

Training for 10 epochs

total forward time : 1.6457819938659668, total backward time : 2.9304211139678955
Train epoch 1: top1=0.5078200101852417%, top2=0.7101199626922607%, top3=0.8143799901008606%, top4=0.8829399943351746%, top5=0.927839994430542%, loss=0.1686025333043933, time=8.575588703155518s
Test epoch 1: top1=0.6186000108718872%, top2=0.8014999628067017%, top3=0.8880999684333801%, top4=0.9376999735832214%, top5=0.9645999670028687%, loss=0.134249168099463, time=1.0585448741912842s
Memory Usage  - Allocated: 48.90 MB, Reserved: 70.00 MB
total forward time : 1.6438982486724854, total backward time : 2.9572885036468506
Train epoch 2: top1=0.6641199588775635%, top2=0.8337399959564209%, top3=0.9092999696731567%, top4=0.9482599496841431%, top5=0.9714999794960022%, loss=0.1192103689019382, time=8.552180528640747s
Test epoch 2: top1=0.6728000044822693%, top2=0.836899995803833%, top3=0.9113999605178833%, top4=0.9497999548912048%, top5=0.9691999554634094%, loss=0.11681944370418787, time=1.

In [56]:
append_to_file(file_name='TRL_report.txt', text=f'########################################')